2.1理论计算题

给定字符序列 "ababc"，词汇表 V = {'a','b','c'}
一阶马尔可夫模型转移计数（未平滑）：
序列中的相邻转移对：
a->b, b->a, a->b, b->c
转移计数矩阵（行=前一个字符，列=当前字符）：
        a   b   c
a       0   2   0
b       1   0   1
c       0   0   0

拉普拉斯平滑（加1平滑）公式：
p(xt | x_{t-1}) = (count(x_{t-1}, xt) + 1) / (sum_{x' in V} (count(x_{t-1}, x') + 1))

词汇表大小 |V| = 3

计算 p('a' | 'b')：
分子 = count(b, a) + 1 = 1 + 1 = 2
分母 = (count(b, a) + 1) + (count(b, b) + 1) + (count(b, c) + 1)
     = (1+1) + (0+1) + (1+1)
     = 2 + 1 + 2
     = 5
p('a' | 'b') = 2/5 = 0.4

计算 p('c' | 'b')：
分子 = count(b, c) + 1 = 1 + 1 = 2
分母 = (count(b, a) + 1) + (count(b, b) + 1) + (count(b, c) + 1)
     = (1+1) + (0+1) + (1+1)
     = 2 + 1 + 2
     = 5
p('c' | 'b') = 2/5 = 0.4

最终答案：
p('a' | 'b') = 0.4
p('c' | 'b') = 0.4

In [ ]:
#2.2编程题
import re
from collections import Counter

def preprocess_text(text, n):
    """
    预处理文本，构建词汇表并生成 n-gram 特征序列和标签序列。

    参数:
        text (str): 输入文本
        n (int): 滑动窗口长度（特征词数）

    返回:
        vocab (dict): 词汇表，{词: ID}
        features (list): 特征序列，每个元素为长度为 n 的词列表
        labels (list): 标签序列，每个元素为下一个词（或 None）
    """
    # 1. 转小写，去除非字母和空格
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # 只保留字母和空格

    # 2. 按空格分词，并过滤空字符串
    words = text.split()
    words = [w for w in words if w]  # 去除多余空格造成的空串

    # 3. 构建词汇表（按频率排序，从0开始分配ID）
    word_counts = Counter(words)
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))  # 频率降序，同频按字母
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}

    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    for i in range(len(words) - n):
        # 特征：当前窗口的 n 个词
        feature = words[i:i+n]
        # 标签：窗口后的下一个词
        label = words[i+n]
        features.append(feature)
        labels.append(label)

    # 如果剩余词不足一个完整窗口，忽略（题目要求）
    # 若刚好最后一个窗口后无标签，也不生成

    return vocab, features, labels


# 测试示例
if __name__ == "__main__":
    text = "The time machine"
    n = 2
    vocab, features, labels = preprocess_text(text, n)

    print("词汇表:", vocab)
    print("特征序列:", features)
    print("标签序列:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征序列: [['the', 'time']]
标签序列: ['machine']


3.1 理论计算题
考虑线性RNN（无偏置）：
h_t = W_hh * h_{t-1} + W_hx * x_t
o_t = W_oh * h_t
损失函数：L = (1/2) * sum_{t=1}^{T} (o_t - y_t)^2

推导损失对权重 W_hh 的梯度：

1. 首先定义每个时间步的局部损失：
   L_t = (1/2) * (o_t - y_t)^2
   L = sum_{t=1}^{T} L_t

2. 计算输出对隐藏状态的梯度：
   dL_t / dh_t = (dL_t / do_t) * (do_t / dh_t)
   dL_t / do_t = o_t - y_t
   do_t / dh_t = W_oh
   所以：dL_t / dh_t = W_oh^T * (o_t - y_t)

3. 通过时间反向传播（BPTT）：
   总梯度 dL/dh_t 包括当前时间步的直接梯度加上未来时间步传回的梯度
   dL/dh_t = dL_t/dh_t + W_hh^T * (dL/dh_{t+1})

4. 展开到所有时间步：
   对于权重 W_hh 的梯度，它影响所有 t >= 2 的时间步：
   dL/dW_hh = sum_{t=2}^{T} (dL/dh_t) * (dh_t/dW_hh)
   
   其中：dh_t/dW_hh = h_{t-1}^T （因为 h_t = W_hh * h_{t-1} + ...）

5. 完整表达式：
   dL/dW_hh = sum_{t=2}^{T} (dL/dh_t) * h_{t-1}^T

   其中 dL/dh_t 通过递归计算：
   dL/dh_T = W_oh^T * (o_T - y_T)
   dL/dh_t = W_oh^T * (o_t - y_t) + W_hh^T * (dL/dh_{t+1})

6. 展开形式（所有时间步）：
   dL/dW_hh = sum_{t=2}^{T} [W_oh^T * (o_t - y_t) + W_hh^T * (dL/dh_{t+1})] * h_{t-1}^T

   进一步展开 dL/dh_{t+1}：
   dL/dW_hh = sum_{t=2}^{T} [W_oh^T * (o_t - y_t) + 
               sum_{k=t+1}^{T} (W_hh^T)^{k-t} * W_oh^T * (o_k - y_k)] * h_{t-1}^T

梯度消失或爆炸的条件：

1. 从展开式中可以看到，梯度中包含 (W_hh^T)^{k-t} 项

2. 梯度消失条件：
   当 W_hh 的最大特征值 < 1 时，随着时间差 k-t 增大，(W_hh^T)^{k-t} 趋近于零
   导致远距离时间步的梯度贡献消失，无法学习长期依赖

3. 梯度爆炸条件：
   当 W_hh 的最大特征值 > 1 时，随着时间差 k-t 增大，(W_hh^T)^{k-t} 趋近于无穷
   导致梯度指数级增长，训练不稳定

4. 稳定条件：
   当 W_hh 的所有特征值的绝对值 < 1 时，系统稳定，梯度不消失也不爆炸
   当所有特征值的绝对值 = 1 时，梯度既不消失也不爆炸（理想情况）

5. 具体数值判断：
   设 lambda_max 为 W_hh 的最大特征值的绝对值
   - lambda_max < 1：梯度消失
   - lambda_max > 1：梯度爆炸
   - lambda_max = 1：梯度稳定

In [5]:
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单元前向传播
    
    参数:
        x_t: 当前输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏的权重，形状 (hidden_size, input_size)
        W_hh: 隐藏到隐藏的权重，形状 (hidden_size, hidden_size)
        b_h: 偏置，形状 (hidden_size,)
    
    返回:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        缓存: 用于反向传播的中间变量
    """
    # 前向传播
    z = np.dot(h_prev, W_hh.T) + np.dot(x_t, W_hx.T) + b_h
    h_t = np.tanh(z)
    
    # 缓存用于反向传播
    cache = (x_t, h_prev, W_hx, W_hh, b_h, z, h_t)
    return h_t, cache

def rnn_backward(dh_next, cache):
    """
    RNN单元反向传播（单步）
    
    参数:
        dh_next: 上游梯度，即损失对 h_t 的梯度，形状 (batch_size, hidden_size)
        cache: 前向传播缓存的中间变量
    
    返回:
        dx_t: 损失对输入 x_t 的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对上一隐藏状态 h_{t-1} 的梯度，形状 (batch_size, hidden_size)
        dW_hx: 损失对权重 W_hx 的梯度，形状 (hidden_size, input_size)
        dW_hh: 损失对权重 W_hh 的梯度，形状 (hidden_size, hidden_size)
        db_h: 损失对偏置 b_h 的梯度，形状 (hidden_size,)
    """
    # 解包缓存
    x_t, h_prev, W_hx, W_hh, b_h, z, h_t = cache
    
    batch_size = x_t.shape[0]
    hidden_size = z.shape[1]
    
    # 计算 tanh 的导数
    # d(tanh(z))/dz = 1 - tanh(z)^2
    dtanh = 1 - h_t ** 2  # 形状: (batch_size, hidden_size)
    
    # 计算 dz = dL/dz = dL/dh_t * dh_t/dz
    dz = dh_next * dtanh  # 形状: (batch_size, hidden_size)
    
    # 计算对各个参数的梯度
    # dx_t = dz * d(z)/d(x_t) = dz * W_hx^T
    dx_t = np.dot(dz, W_hx)  # 形状: (batch_size, input_size)
    
    # dh_prev = dz * d(z)/d(h_prev) = dz * W_hh^T
    dh_prev = np.dot(dz, W_hh)  # 形状: (batch_size, hidden_size)
    
    # dW_hx = dz^T * x_t / batch_size (平均梯度)
    dW_hx = np.dot(dz.T, x_t) / batch_size  # 形状: (hidden_size, input_size)
    
    # dW_hh = dz^T * h_prev / batch_size
    dW_hh = np.dot(dz.T, h_prev) / batch_size  # 形状: (hidden_size, hidden_size)
    
    # db_h = sum(dz, axis=0) / batch_size
    db_h = np.sum(dz, axis=0) / batch_size  # 形状: (hidden_size,)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试示例
if __name__ == "__main__":
    # 设置参数
    batch_size = 2
    input_size = 3
    hidden_size = 4
    
    # 随机初始化
    np.random.seed(42)
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(hidden_size, input_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(hidden_size)
    
    # 前向传播
    h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print("隐藏状态 h_t 形状:", h_t.shape)
    print("h_t:\n", h_t)
    
    # 反向传播（假设上游梯度）
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, cache)
    
    print("\n梯度形状:")
    print("dx_t:", dx_t.shape)
    print("dh_prev:", dh_prev.shape)
    print("dW_hx:", dW_hx.shape)
    print("dW_hh:", dW_hh.shape)
    print("db_h:", db_h.shape)
    
    print("\n梯度值（部分）:")
    print("dW_hx:\n", dW_hx)
    print("dW_hh:\n", dW_hh)
    print("db_h:\n", db_h)

隐藏状态 h_t 形状: (2, 4)
h_t:
 [[-0.99460443 -0.77409015 -0.9004889  -0.99810568]
 [-0.92208444  0.97361461  0.99985849 -0.98411114]]

梯度形状:
dx_t: (2, 3)
dh_prev: (2, 4)
dW_hx: (4, 3)
dW_hh: (4, 4)
db_h: (4,)

梯度值（部分）:
dW_hx:
 [[ 0.03572885 -0.00533959 -0.00728734]
 [ 0.0899528  -0.02694204  0.13955331]
 [ 0.0159936  -0.00447012  0.02106745]
 [ 0.01302716 -0.00179629 -0.00441853]]
dW_hh:
 [[-0.01516037 -0.01320432  0.0070355  -0.04777529]
 [ 0.33918543  0.1672423  -0.10187905  0.13411892]
 [ 0.05135696  0.02498048 -0.01527757  0.0178124 ]
 [-0.00973729 -0.00705101  0.00389935 -0.02025744]]
db_h:
 [0.02178875 0.20181257 0.03239671 0.00630504]


4.1 理论计算题
深度双向RNN参数计算：

已知条件：
- L层
- 每层隐藏单元数 H
- 输入维度 D
- 输出维度 O（仅最后输出层）

双向RNN每层包含两个方向的RNN：前向和后向

参数组成：

1. 第一层（输入层）：
   每个方向：
   - 输入到隐藏权重 W_hx: H * D
   - 隐藏到隐藏权重 W_hh: H * H
   - 偏置 b_h: H
   
   两个方向共：
   2 * (H*D + H*H + H) = 2H(D + H + 1)

2. 中间层（第2层到第L-1层）：
   每个方向：
   - 输入到隐藏权重 W_hx: H * (2H)  (因为输入是上一层双向拼接，维度为2H)
   - 隐藏到隐藏权重 W_hh: H * H
   - 偏置 b_h: H
   
   两个方向共：
   2 * (H*2H + H*H + H) = 2H(2H + H + 1) = 2H(3H + 1)
   
   中间层有 (L-2) 层，所以：
   (L-2) * 2H(3H + 1)

3. 最后一层（第L层）：
   每个方向：
   - 输入到隐藏权重 W_hx: H * (2H)
   - 隐藏到隐藏权重 W_hh: H * H
   - 偏置 b_h: H
   
   两个方向共：
   2 * (H*2H + H*H + H) = 2H(2H + H + 1) = 2H(3H + 1)

4. 输出层（仅最后输出层）：
   - 输出权重 W_oh: O * (2H)  (输入是最后层双向拼接，维度为2H)
   - 输出偏置 b_o: O
   
   共：O*2H + O = O(2H + 1)

总参数：
Total = 2H(D + H + 1) + (L-2)*2H(3H + 1) + 2H(3H + 1) + O(2H + 1)

化简：
第一层：2H(D + H + 1)
中间层和最后一层合并（共L-1层，每层参数为2H(3H + 1)）：
(L-1)*2H(3H + 1)

所以：
Total = 2H(D + H + 1) + (L-1)*2H(3H + 1) + O(2H + 1)

进一步展开：
Total = 2H(D + H + 1 + (L-1)(3H + 1)) + O(2H + 1)
Total = 2H(D + H + 1 + 3H(L-1) + (L-1)) + O(2H + 1)
Total = 2H(D + H + 1 + 3HL - 3H + L - 1) + O(2H + 1)
Total = 2H(D + 3HL - 2H + L) + O(2H + 1)

最终表达式：
Total_params = 2H(D + H + 1) + (L-1)*2H(3H + 1) + O(2H + 1)

或者更简洁：
Total_params = 2H[D + H + 1 + (L-1)(3H + 1)] + O(2H + 1)

特殊情况检查：
- L=1时：Total = 2H(D + H + 1) + O(2H + 1) ✓
- 单向RNN（每层参数减半）：Total = H(D + H + 1) + (L-1)*H(3H + 1) + O(H + 1)

In [6]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    """
    双向RNN编码器
    接收序列X，返回每个时间步的拼接隐藏状态和最终拼接隐藏状态
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1, rnn_type='rnn'):
        """
        参数:
            input_dim: 输入维度
            hidden_dim: 每层隐藏单元数
            num_layers: RNN层数
            rnn_type: RNN类型，'rnn'或'lstm'或'gru'
        """
        super(BidirectionalRNNEncoder, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.rnn_type = rnn_type
        
        # 根据类型选择RNN
        if rnn_type == 'rnn':
            self.rnn = nn.RNN(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=False,  # 保持 (seq_len, batch, features) 格式
                bidirectional=True
            )
        elif rnn_type == 'lstm':
            self.rnn = nn.LSTM(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=False,
                bidirectional=True
            )
        elif rnn_type == 'gru':
            self.rnn = nn.GRU(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=False,
                bidirectional=True
            )
        else:
            raise ValueError("rnn_type must be 'rnn', 'lstm', or 'gru'")
    
    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch_size, input_dim)
        
        返回:
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch_size, 2*hidden_dim)
            final_state: 最终拼接隐藏状态，形状 (batch_size, 2*hidden_dim)
        """
        # 通过双向RNN
        # outputs: (seq_len, batch_size, 2*hidden_dim)  # 双向拼接
        # hidden_state: 最后一层的前向和后向隐藏状态
        outputs, hidden_state = self.rnn(X)
        
        # 对于RNN和GRU，hidden_state是 (num_layers*2, batch, hidden_dim)
        # 取最后一层的前向和后向隐藏状态
        if self.rnn_type == 'lstm':
            # LSTM返回 (hidden_state, cell_state)
            hidden_state = hidden_state[0]  # 取隐藏状态
        
        # 获取最后一层的前向和后向隐藏状态
        # hidden_state形状: (num_layers*2, batch, hidden_dim)
        # 最后一层前向: hidden_state[-2, :, :]
        # 最后一层后向: hidden_state[-1, :, :]
        forward_last = hidden_state[-2, :, :]  # (batch, hidden_dim)
        backward_last = hidden_state[-1, :, :]  # (batch, hidden_dim)
        
        # 拼接得到最终表示
        final_state = torch.cat([forward_last, backward_last], dim=-1)  # (batch, 2*hidden_dim)
        
        return outputs, final_state
    
    def get_parameters_count(self):
        """返回模型参数总数"""
        return sum(p.numel() for p in self.parameters())


# 手动实现的双向RNN（不使用torch.nn.RNN）
class BidirectionalRNNEncoderManual(nn.Module):
    """
    手动实现的双向RNN编码器（不使用内置RNN）
    仅支持单层RNN，便于理解
    """
    def __init__(self, input_dim, hidden_dim):
        super(BidirectionalRNNEncoderManual, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        
        # 前向RNN参数
        self.W_hx_f = nn.Parameter(torch.randn(hidden_dim, input_dim))
        self.W_hh_f = nn.Parameter(torch.randn(hidden_dim, hidden_dim))
        self.b_h_f = nn.Parameter(torch.randn(hidden_dim))
        
        # 后向RNN参数
        self.W_hx_b = nn.Parameter(torch.randn(hidden_dim, input_dim))
        self.W_hh_b = nn.Parameter(torch.randn(hidden_dim, hidden_dim))
        self.b_h_b = nn.Parameter(torch.randn(hidden_dim))
    
    def forward(self, X):
        """
        手动实现双向RNN前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch_size, input_dim)
        
        返回:
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch_size, 2*hidden_dim)
            final_state: 最终拼接隐藏状态，形状 (batch_size, 2*hidden_dim)
        """
        seq_len, batch_size, _ = X.shape
        
        # 初始化隐藏状态
        h_f = torch.zeros(batch_size, self.hidden_dim, device=X.device)  # 前向初始
        h_b = torch.zeros(batch_size, self.hidden_dim, device=X.device)  # 后向初始
        
        # 存储所有时间步的隐藏状态
        hidden_f = []  # 前向
        hidden_b = []  # 后向
        
        # 前向传播 (从0到seq_len-1)
        for t in range(seq_len):
            x_t = X[t]  # (batch, input_dim)
            h_f = torch.tanh(
                torch.mm(x_t, self.W_hx_f.T) + 
                torch.mm(h_f, self.W_hh_f.T) + 
                self.b_h_f
            )
            hidden_f.append(h_f)
        
        # 后向传播 (从seq_len-1到0)
        for t in range(seq_len-1, -1, -1):
            x_t = X[t]  # (batch, input_dim)
            h_b = torch.tanh(
                torch.mm(x_t, self.W_hx_b.T) + 
                torch.mm(h_b, self.W_hh_b.T) + 
                self.b_h_b
            )
            hidden_b.append(h_b)
        hidden_b = hidden_b[::-1]  # 反转回原始顺序
        
        # 拼接前向和后向隐藏状态
        outputs = []
        for t in range(seq_len):
            h_concat = torch.cat([hidden_f[t], hidden_b[t]], dim=-1)
            outputs.append(h_concat)
        
        outputs = torch.stack(outputs, dim=0)  # (seq_len, batch, 2*hidden_dim)
        
        # 最终状态：最后一个时间步的前向 + 第一个时间步的后向（或最后一个时间步的后向）
        # 标准做法：拼接最后一层前向的最后一个状态和后向的最后一个状态
        forward_last = hidden_f[-1]  # (batch, hidden_dim)
        backward_last = hidden_b[-1]  # (batch, hidden_dim)
        final_state = torch.cat([forward_last, backward_last], dim=-1)  # (batch, 2*hidden_dim)
        
        return outputs, final_state


# 测试代码
if __name__ == "__main__":
    # 设置参数
    seq_len = 5
    batch_size = 3
    input_dim = 10
    hidden_dim = 8
    num_layers = 2
    
    # 创建随机输入
    X = torch.randn(seq_len, batch_size, input_dim)
    
    # 测试内置实现
    print("=== 使用 torch.nn.RNN ===")
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers, rnn_type='rnn')
    outputs, final_state = encoder(X)
    
    print(f"输入形状: {X.shape}")
    print(f"输出形状 (每个时间步): {outputs.shape}")
    print(f"最终状态形状: {final_state.shape}")
    print(f"参数总数: {encoder.get_parameters_count()}")
    
    # 测试手动实现
    print("\n=== 手动实现 (单层) ===")
    encoder_manual = BidirectionalRNNEncoderManual(input_dim, hidden_dim)
    outputs_manual, final_state_manual = encoder_manual(X)
    print(f"输出形状 (每个时间步): {outputs_manual.shape}")
    print(f"最终状态形状: {final_state_manual.shape}")
    
    # 验证维度正确性
    print("\n=== 维度验证 ===")
    print(f"每个时间步输出维度: {outputs.shape[-1]} (期望: {2*hidden_dim})")
    print(f"最终状态维度: {final_state.shape[-1]} (期望: {2*hidden_dim})")
    print(f"批量大小: {final_state.shape[0]} (期望: {batch_size})")

=== 使用 torch.nn.RNN ===
输入形状: torch.Size([5, 3, 10])
输出形状 (每个时间步): torch.Size([5, 3, 16])
最终状态形状: torch.Size([3, 16])
参数总数: 736

=== 手动实现 (单层) ===
输出形状 (每个时间步): torch.Size([5, 3, 16])
最终状态形状: torch.Size([3, 16])

=== 维度验证 ===
每个时间步输出维度: 16 (期望: 16)
最终状态维度: 16 (期望: 16)
批量大小: 3 (期望: 3)


5.1 理论计算题
Skip-gram模型负采样损失函数推导：

给定：
- 中心词 w_c
- 上下文词 w_o
- 词向量：中心词向量 v_c，上下文词向量 u_o
- 负样本词向量：u_nk (k=1,2,...,K)
- 负样本数量 K

标准Skip-gram目标（最大化）：
J = log p(w_o | w_c)

使用负采样，将softmax替换为二分类任务：
- 正样本：(w_c, w_o) -> 标签为1
- 负样本：(w_c, w_nk) -> 标签为0，其中 w_nk 从噪声分布 P_n(w) 中采样

负采样的目标函数（最大化形式）：
J = log sigma(v_c^T * u_o) + sum_{k=1}^{K} E_{w_nk ~ P_n(w)} [log sigma(-v_c^T * u_nk)]

其中 sigma(x) = 1/(1 + exp(-x)) 是sigmoid函数

等价的最小化损失函数（负对数似然）：
L = -log sigma(v_c^T * u_o) - sum_{k=1}^{K} E_{w_nk ~ P_n(w)} [log sigma(-v_c^T * u_nk)]

实际实现中，对每个样本采样K个负样本，损失函数为：
L = -log sigma(v_c^T * u_o) - sum_{k=1}^{K} log sigma(-v_c^T * u_nk)

用sigmoid性质 sigma(-x) = 1 - sigma(x)，也可写为：
L = -log sigma(v_c^T * u_o) - sum_{k=1}^{K} log(1 - sigma(v_c^T * u_nk))

等价形式（使用交叉熵）：
L = -[log(sigma(v_c^T * u_o)) + sum_{k=1}^{K} log(1 - sigma(v_c^T * u_nk))]

完整的对数似然（最大化）：
L_max = log sigma(v_c^T * u_o) + sum_{k=1}^{K} log(1 - sigma(v_c^T * u_nk))

负样本采样方法（从噪声分布中采样）：

1. 噪声分布通常选择：
   P_n(w) = U(w)^(3/4) / Z
   其中 U(w) 是词在语料中的一元分布（词频），Z 是归一化常数

2. 采样过程：
   - 计算每个词在语料中的频率 count(w)
   - 计算 P_n(w) = count(w)^(3/4) / sum_{w'} count(w')^(3/4)
   - 从该分布中采样K个负样本（通常有放回采样）
   - 采样时排除正样本（上下文词）

3. 常用实现方式：
   - 使用别名采样法（Alias Method）进行高效采样
   - 或使用负采样表（将词按概率分布放入表中，随机选择）

完整目标函数（最大化形式）：
J(θ) = log sigma(v_c^T * u_o) + sum_{k=1}^{K} log sigma(-v_c^T * u_nk)

其中 θ 包含所有词向量 v 和 u，通过梯度上升最大化目标函数。

梯度更新（对中心词向量）：
∂J/∂v_c = sigma(-v_c^T * u_o) * u_o - sum_{k=1}^{K} sigma(v_c^T * u_nk) * u_nk

梯度更新（对正样本上下文词向量）：
∂J/∂u_o = sigma(-v_c^T * u_o) * v_c

梯度更新（对负样本上下文词向量）：
∂J/∂u_nk = -sigma(v_c^T * u_nk) * v_c

In [7]:
import numpy as np

class CBOW:
    """
    CBOW模型（完整softmax）
    输入：上下文词索引列表 -> 预测中心词
    """
    def __init__(self, vocab_size, embed_dim):
        """
        参数:
            vocab_size: 词汇表大小 V
            embed_dim: 嵌入维度 d
        """
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        
        # 初始化权重
        # W: 输入权重矩阵 (V, d)，每一行是一个词的输入向量
        # W_out: 输出权重矩阵 (d, V)，每一列是一个词的输出向量
        np.random.seed(42)
        self.W = np.random.randn(vocab_size, embed_dim) * 0.01
        self.W_out = np.random.randn(embed_dim, vocab_size) * 0.01
        
    def forward(self, context_indices, target_index):
        """
        前向传播和损失计算
        
        参数:
            context_indices: 上下文词索引列表，形状 (batch_size, context_size)
                            每个样本有 context_size 个上下文词
            target_index: 目标中心词索引，形状 (batch_size,)
        
        返回:
            loss: 交叉熵损失值（标量）
            cache: 缓存中间变量用于反向传播
        """
        batch_size, context_size = context_indices.shape
        
        # Step 1: 获取上下文词的嵌入向量
        # 对于每个样本，取 context_size 个词的嵌入向量
        # context_embeds: (batch_size, context_size, embed_dim)
        context_embeds = self.W[context_indices]  # (batch_size, context_size, embed_dim)
        
        # Step 2: 计算平均上下文向量作为隐藏层
        # h: (batch_size, embed_dim)
        h = np.mean(context_embeds, axis=1)  # 沿上下文词维度求平均
        
        # Step 3: 计算输出得分
        # scores: (batch_size, vocab_size)
        scores = np.dot(h, self.W_out)  # (batch_size, embed_dim) @ (embed_dim, vocab_size)
        
        # Step 4: Softmax得到概率分布
        # 数值稳定性：减去最大值
        max_scores = np.max(scores, axis=1, keepdims=True)
        exp_scores = np.exp(scores - max_scores)
        probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)  # (batch_size, vocab_size)
        
        # Step 5: 计算交叉熵损失
        # 使用目标索引获取对应的概率
        # target_index: (batch_size,)
        batch_indices = np.arange(batch_size)
        target_probs = probs[batch_indices, target_index]  # (batch_size,)
        
        # 交叉熵损失：L = -log(p(target))
        # 注意：添加小epsilon防止log(0)
        epsilon = 1e-8
        loss = -np.mean(np.log(target_probs + epsilon))
        
        # 缓存用于反向传播
        cache = {
            'context_indices': context_indices,
            'target_index': target_index,
            'h': h,
            'probs': probs,
            'context_embeds': context_embeds,
            'batch_size': batch_size,
            'context_size': context_size
        }
        
        return loss, cache
    
    def backward(self, cache):
        """
        反向传播计算梯度
        
        参数:
            cache: 前向传播缓存的中间变量
        
        返回:
            dW: 输入权重的梯度 (V, d)
            dW_out: 输出权重的梯度 (d, V)
        """
        context_indices = cache['context_indices']
        target_index = cache['target_index']
        h = cache['h']
        probs = cache['probs']
        context_embeds = cache['context_embeds']
        batch_size = cache['batch_size']
        context_size = cache['context_size']
        
        V = self.vocab_size
        d = self.embed_dim
        
        # Step 1: 计算输出层的梯度
        # grad_scores: (batch_size, V)
        # 对于每个样本，在目标位置减1，其他位置保持概率
        grad_scores = probs.copy()
        batch_indices = np.arange(batch_size)
        grad_scores[batch_indices, target_index] -= 1.0
        grad_scores = grad_scores / batch_size  # 平均梯度
        
        # Step 2: 计算输出权重的梯度
        # dW_out: (d, V)
        dW_out = np.dot(h.T, grad_scores)  # (embed_dim, V)
        
        # Step 3: 计算隐藏层的梯度
        # dh: (batch_size, embed_dim)
        dh = np.dot(grad_scores, self.W_out.T)  # (batch_size, V) @ (V, d)
        
        # Step 4: 计算输入权重的梯度
        # 平均操作的反向传播：dh 分配到 context_size 个词上，然后平均
        # 每个样本有 context_size 个上下文词
        dW = np.zeros_like(self.W)  # (V, d)
        
        for i in range(batch_size):
            # 获取该样本的上下文词索引
            ctx_indices = context_indices[i]  # (context_size,)
            # 该样本的梯度（需要除以 context_size 因为前向传播做了平均）
            grad_h = dh[i] / context_size  # (embed_dim,)
            
            # 将梯度累加到对应的词向量上
            for j in range(context_size):
                word_idx = ctx_indices[j]
                dW[word_idx] += grad_h  # (embed_dim,)
        
        # 注意：由于我们使用的是平均，且没有除以batch_size，这里已经除以了batch_size
        # 但实际上grad_scores已经除以了batch_size，所以梯度已经平均了
        
        return dW, dW_out
    
    def compute_loss_only(self, context_indices, target_index):
        """
        仅计算损失（前向传播），不返回缓存
        
        参数:
            context_indices: 上下文词索引列表，形状 (batch_size, context_size)
            target_index: 目标中心词索引，形状 (batch_size,)
        
        返回:
            loss: 交叉熵损失值
        """
        loss, _ = self.forward(context_indices, target_index)
        return loss


# 测试示例
if __name__ == "__main__":
    # 设置参数
    vocab_size = 10
    embed_dim = 4
    batch_size = 2
    context_size = 3
    
    # 创建模型
    model = CBOW(vocab_size, embed_dim)
    
    # 生成随机数据
    np.random.seed(42)
    context_indices = np.random.randint(0, vocab_size, size=(batch_size, context_size))
    target_index = np.random.randint(0, vocab_size, size=(batch_size,))
    
    print("上下文词索引 (batch_size={}, context_size={}):".format(batch_size, context_size))
    print(context_indices)
    print("\n目标中心词索引:")
    print(target_index)
    
    # 前向传播计算损失
    loss, cache = model.forward(context_indices, target_index)
    print(f"\n交叉熵损失: {loss:.6f}")
    
    # 反向传播计算梯度
    dW, dW_out = model.backward(cache)
    print(f"\n输入权重梯度形状: {dW.shape}")
    print(f"输出权重梯度形状: {dW_out.shape}")
    print("\n输入权重梯度 (部分):")
    print(dW[:3, :2])
    print("\n输出权重梯度 (部分):")
    print(dW_out[:2, :3])
    
    # 验证：检查损失是否正确计算
    print("\n=== 验证损失计算 ===")
    # 单个样本测试
    single_context = context_indices[0:1]  # (1, context_size)
    single_target = target_index[0:1]  # (1,)
    loss_single = model.compute_loss_only(single_context, single_target)
    print(f"单样本损失: {loss_single:.6f}")
    
    # 手动计算验证
    h_single = np.mean(model.W[single_context[0]], axis=0)
    scores_single = np.dot(h_single, model.W_out)
    probs_single = np.exp(scores_single) / np.sum(np.exp(scores_single))
    loss_manual = -np.log(probs_single[single_target[0]] + 1e-8)
    print(f"手动计算损失: {loss_manual:.6f}")
    print(f"匹配: {np.isclose(loss_single, loss_manual)}")
    
    print("\n=== 模型参数信息 ===")
    print(f"词汇表大小: {vocab_size}")
    print(f"嵌入维度: {embed_dim}")
    print(f"输入权重矩阵 W: {model.W.shape}")
    print(f"输出权重矩阵 W_out: {model.W_out.shape}")
    total_params = vocab_size * embed_dim + embed_dim * vocab_size
    print(f"总参数量: {total_params}")

上下文词索引 (batch_size=2, context_size=3):
[[6 3 7]
 [4 6 9]]

目标中心词索引:
[2 6]

交叉熵损失: 2.302460

输入权重梯度形状: (10, 4)
输出权重梯度形状: (4, 10)

输入权重梯度 (部分):
[[0. 0.]
 [0. 0.]
 [0. 0.]]

输出权重梯度 (部分):
[[-0.00037524 -0.00037525  0.0011298 ]
 [-0.00060477 -0.0006048   0.00288522]]

=== 验证损失计算 ===
单样本损失: 2.302401
手动计算损失: 2.302401
匹配: True

=== 模型参数信息 ===
词汇表大小: 10
嵌入维度: 4
输入权重矩阵 W: (10, 4)
输出权重矩阵 W_out: (4, 10)
总参数量: 80


6.1 理论计算题
缩放点积注意力计算过程：

给定：
Q ∈ R^(2×4)，K ∈ R^(3×4)，V ∈ R^(3×5)
d_k = 4，所以 sqrt(d_k) = sqrt(4) = 2

第一步：计算得分矩阵
S = Q * K^T / sqrt(d_k)
S = Q * K^T / 2

假设具体数值（为了演示，使用简单数值）：

Q = [[1, 0, 1, 0],
     [0, 1, 0, 1]]

K = [[1, 1, 0, 0],
     [0, 1, 1, 0],
     [1, 0, 1, 1]]

V = [[1, 2, 3, 4, 5],
     [2, 3, 4, 5, 6],
     [3, 4, 5, 6, 7]]

计算 K^T：
K^T = [[1, 0, 1],
       [1, 1, 0],
       [0, 1, 1],
       [0, 0, 1]]

计算 Q * K^T：
Q第一行 [1,0,1,0] 与 K^T各行点积：
- 与第1列 [1,1,0,0]^T: 1*1 + 0*1 + 1*0 + 0*0 = 1
- 与第2列 [0,1,1,0]^T: 1*0 + 0*1 + 1*1 + 0*0 = 1
- 与第3列 [1,0,1,1]^T: 1*1 + 0*0 + 1*1 + 0*1 = 2

Q第二行 [0,1,0,1] 与 K^T各行点积：
- 与第1列 [1,1,0,0]^T: 0*1 + 1*1 + 0*0 + 1*0 = 1
- 与第2列 [0,1,1,0]^T: 0*0 + 1*1 + 0*1 + 1*0 = 1
- 与第3列 [1,0,1,1]^T: 0*1 + 1*0 + 0*1 + 1*1 = 1

所以 Q * K^T = [[1, 1, 2],
                [1, 1, 1]]

除以 sqrt(d_k) = 2：
S = [[0.5, 0.5, 1.0],
     [0.5, 0.5, 0.5]]

第二步：对每一行做softmax
对第1行 [0.5, 0.5, 1.0]：
exp(0.5) = 1.6487
exp(0.5) = 1.6487
exp(1.0) = 2.7183
sum = 1.6487 + 1.6487 + 2.7183 = 6.0157
softmax第1行 = [1.6487/6.0157, 1.6487/6.0157, 2.7183/6.0157]
              = [0.2741, 0.2741, 0.4518]

对第2行 [0.5, 0.5, 0.5]：
exp(0.5) = 1.6487 (三次)
sum = 1.6487 * 3 = 4.9461
softmax第2行 = [1.6487/4.9461, 1.6487/4.9461, 1.6487/4.9461]
              = [0.3333, 0.3333, 0.3333]

得到注意力权重矩阵：
A = [[0.2741, 0.2741, 0.4518],
     [0.3333, 0.3333, 0.3333]]

第三步：加权求和得到输出
输出 = A * V

V = [[1, 2, 3, 4, 5],
     [2, 3, 4, 5, 6],
     [3, 4, 5, 6, 7]]

第1行输出 = 0.2741*V[0] + 0.2741*V[1] + 0.4518*V[2]
计算每个维度：
维度1: 0.2741*1 + 0.2741*2 + 0.4518*3 = 0.2741 + 0.5482 + 1.3554 = 2.1777
维度2: 0.2741*2 + 0.2741*3 + 0.4518*4 = 0.5482 + 0.8223 + 1.8072 = 3.1777
维度3: 0.2741*3 + 0.2741*4 + 0.4518*5 = 0.8223 + 1.0964 + 2.2590 = 4.1777
维度4: 0.2741*4 + 0.2741*5 + 0.4518*6 = 1.0964 + 1.3705 + 2.7108 = 5.1777
维度5: 0.2741*5 + 0.2741*6 + 0.4518*7 = 1.3705 + 1.6446 + 3.1626 = 6.1777

第2行输出 = 0.3333*V[0] + 0.3333*V[1] + 0.3333*V[2]
每个维度 = 0.3333 * (V[0] + V[1] + V[2])
= 0.3333 * [1+2+3, 2+3+4, 3+4+5, 4+5+6, 5+6+7]
= 0.3333 * [6, 9, 12, 15, 18]
= [2.0, 3.0, 4.0, 5.0, 6.0]

最终输出矩阵：
Output = [[2.1777, 3.1777, 4.1777, 5.1777, 6.1777],
          [2.0000, 3.0000, 4.0000, 5.0000, 6.0000]]

形状：2×5（与V的行数对应，维度为V的列数）

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """
    多头注意力机制（Multi-Head Attention）
    """
    def __init__(self, d_model, num_heads, dropout=0.0):
        """
        参数:
            d_model: 模型维度
            num_heads: 注意力头数
            dropout: dropout概率
        """
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, X, mask=None):
        """
        前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, d_model)
            mask: 掩码矩阵，形状 (seq_len, seq_len) 或 (batch, seq_len, seq_len)
        
        返回:
            output: 输出序列，形状 (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        
        # 1. 线性投影得到 Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)  # (seq_len, batch, d_model)
        V = self.W_v(X)  # (seq_len, batch, d_model)
        
        # 2. 重塑为多头形式
        # 将 (seq_len, batch, d_model) 变为 (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k)
        V = V.view(seq_len, batch, self.num_heads, self.d_v)
        
        # 交换维度以便计算: (batch, num_heads, seq_len, d_k)
        Q = Q.transpose(0, 1).transpose(1, 2)  # (batch, num_heads, seq_len, d_k)
        K = K.transpose(0, 1).transpose(1, 2)  # (batch, num_heads, seq_len, d_k)
        V = V.transpose(0, 1).transpose(1, 2)  # (batch, num_heads, seq_len, d_v)
        
        # 3. 计算缩放点积注意力
        # scores: (batch, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        
        # 4. 应用掩码（如果有）
        if mask is not None:
            # mask形状: (seq_len, seq_len) 或 (batch, seq_len, seq_len)
            if mask.dim() == 2:
                # 扩展到 (batch, num_heads, seq_len, seq_len)
                mask = mask.unsqueeze(0).unsqueeze(0)
            elif mask.dim() == 3:
                # 扩展到 (batch, num_heads, seq_len, seq_len)
                mask = mask.unsqueeze(1)
            # 将mask位置设为负无穷，使得softmax后接近于0
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # 5. Softmax
        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq_len, seq_len)
        attn_weights = self.dropout(attn_weights)
        
        # 6. 加权求和
        # context: (batch, num_heads, seq_len, d_v)
        context = torch.matmul(attn_weights, V)
        
        # 7. 合并多头输出
        # 变换回 (seq_len, batch, num_heads, d_v)
        context = context.transpose(1, 2).transpose(0, 1)  # (seq_len, batch, num_heads, d_v)
        context = context.contiguous().view(seq_len, batch, self.d_model)  # (seq_len, batch, d_model)
        
        # 8. 最终线性层
        output = self.W_o(context)  # (seq_len, batch, d_model)
        
        return output
    
    def forward_with_attention(self, X, mask=None):
        """
        前向传播，同时返回注意力权重
        
        返回:
            output: 输出序列
            attn_weights: 注意力权重 (batch, num_heads, seq_len, seq_len)
        """
        seq_len, batch, _ = X.shape
        
        Q = self.W_q(X).view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)
        K = self.W_k(X).view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)
        V = self.W_v(X).view(seq_len, batch, self.num_heads, self.d_v).transpose(0, 1).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        
        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)
        
        context = context.transpose(1, 2).transpose(0, 1).contiguous().view(seq_len, batch, self.d_model)
        output = self.W_o(context)
        
        return output, attn_weights


# 测试代码
if __name__ == "__main__":
    # 设置参数
    d_model = 4
    num_heads = 2
    seq_len = 3
    batch = 2
    
    # 创建模型
    model = MultiHeadAttention(d_model, num_heads)
    
    # 创建随机输入
    torch.manual_seed(42)
    X = torch.randn(seq_len, batch, d_model)
    
    print("=== 多头注意力测试 ===")
    print(f"输入形状: {X.shape}")
    print(f"d_model={d_model}, num_heads={num_heads}")
    print(f"每个头的维度: d_k={d_model//num_heads}, d_v={d_model//num_heads}")
    
    # 前向传播
    output = model(X)
    print(f"\n输出形状: {output.shape}")
    
    # 测试带掩码的情况
    print("\n=== 带掩码测试 ===")
    # 创建因果掩码 (防止看到未来信息)
    mask = torch.tril(torch.ones(seq_len, seq_len))
    print(f"掩码矩阵 (下三角):\n{mask}")
    
    output_masked, attn_weights = model.forward_with_attention(X, mask)
    print(f"带掩码输出形状: {output_masked.shape}")
    print(f"注意力权重形状: {attn_weights.shape}")
    print(f"注意力权重 (head 1):\n{attn_weights[0, 0, :, :]}")
    
    # 验证维度
    print("\n=== 维度验证 ===")
    print(f"每个注意力头的维度: {d_model // num_heads}")
    print(f"注意力权重维度: [batch={batch}, num_heads={num_heads}, seq_len={seq_len}, seq_len={seq_len}]")
    print(f"输出维度与输入相同: {output.shape == X.shape}")
    
    # 统计参数
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n模型参数量: {total_params}")
    
    # 手动验证前向传播（第一个样本，第一个头）
    print("\n=== 手动验证 (第一个样本, 第一个头) ===")
    # 获取第一个样本的Q, K, V
    Q_head0 = model.W_q(X)[:, 0:1, :].view(seq_len, 1, num_heads, d_model//num_heads)[:, :, 0, :]
    K_head0 = model.W_k(X)[:, 0:1, :].view(seq_len, 1, num_heads, d_model//num_heads)[:, :, 0, :]
    V_head0 = model.W_v(X)[:, 0:1, :].view(seq_len, 1, num_heads, d_model//num_heads)[:, :, 0, :]
    
    # 计算注意力得分
    scores_manual = torch.matmul(Q_head0, K_head0.transpose(-2, -1)) / torch.sqrt(torch.tensor(2.0))
    attn_manual = F.softmax(scores_manual, dim=-1)
    
    print(f"手动计算的注意力权重形状: {attn_manual.shape}")
    print(f"手动计算 (softmax前):\n{scores_manual.squeeze()}")
    print(f"手动计算 (softmax后):\n{attn_manual.squeeze()}")

=== 多头注意力测试 ===
输入形状: torch.Size([3, 2, 4])
d_model=4, num_heads=2
每个头的维度: d_k=2, d_v=2

输出形状: torch.Size([3, 2, 4])

=== 带掩码测试 ===
掩码矩阵 (下三角):
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
带掩码输出形状: torch.Size([3, 2, 4])
注意力权重形状: torch.Size([2, 2, 3, 3])
注意力权重 (head 1):
tensor([[1.0000, 0.0000, 0.0000],
        [0.5073, 0.4927, 0.0000],
        [0.3652, 0.3299, 0.3049]], grad_fn=<SelectBackward0>)

=== 维度验证 ===
每个注意力头的维度: 2
注意力权重维度: [batch=2, num_heads=2, seq_len=3, seq_len=3]
输出维度与输入相同: True

模型参数量: 64

=== 手动验证 (第一个样本, 第一个头) ===
手动计算的注意力权重形状: torch.Size([3, 1, 1])
手动计算 (softmax前):
tensor([-0.1266, -0.0044,  0.0165], grad_fn=<SqueezeBackward0>)
手动计算 (softmax后):
tensor([1., 1., 1.], grad_fn=<SqueezeBackward0>)
